In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Pass Transpiler berbasis AI

Pass Transpiler berbasis AI adalah pass yang berfungsi sebagai pengganti langsung dari pass Qiskit "tradisional" untuk beberapa tugas transpilasi. Pass ini sering menghasilkan hasil yang lebih baik dibandingkan algoritma heuristik yang ada (seperti depth dan jumlah CNOT yang lebih rendah), tetapi juga jauh lebih cepat daripada algoritma optimasi seperti solver Boolean satisfiability. Pass Transpiler AI berjalan di lingkungan lokal kamu.

> **Note:** Pass Transpiler berbasis AI masih dalam status rilis beta dan dapat berubah sewaktu-waktu.
>     Jika kamu punya masukan atau ingin menghubungi tim developer, silakan gunakan [channel Qiskit Slack Workspace ini](https://qiskit.slack.com/archives/C06KF8YHUAU).

Pass yang tersedia saat ini adalah sebagai berikut:

**Pass Routing**
 - `AIRouting`: Pemilihan layout dan routing Circuit

**Pass sintesis Circuit**
 - `AICliffordSynthesis`: Sintesis Circuit Clifford
 - `AILinearFunctionSynthesis`: Sintesis Circuit fungsi linear
 - `AIPermutationSynthesis`: Sintesis Circuit permutasi

Untuk menggunakan pass Transpiler AI, pertama instal paket `qiskit-ibm-transpiler`. Kunjungi [dokumentasi API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) untuk mendapatkan informasi lebih lanjut tentang berbagai opsi yang tersedia.

## Jalankan pass Transpiler AI secara lokal atau di cloud
Pertama instal `qiskit-ibm-transpiler` dengan beberapa dependensi tambahan sebagai berikut:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Setelah menginstal dependensi tambahan, mode default untuk menjalankan pass Transpiler berbasis AI adalah menggunakan mesin lokal kamu.

## Pass routing AI
Pass `AIRouting` berfungsi sebagai stage layout sekaligus stage routing. Pass ini dapat digunakan di dalam `PassManager` sebagai berikut:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Di sini, `backend` menentukan coupling map mana yang akan digunakan untuk routing, `optimization_level` (1, 2, atau 3) menentukan seberapa besar usaha komputasi yang digunakan dalam prosesnya (nilai lebih tinggi biasanya menghasilkan hasil yang lebih baik tetapi membutuhkan waktu lebih lama), dan `layout_mode` menentukan cara menangani pemilihan layout.
`layout_mode` mencakup opsi berikut:

- `keep`: Mode ini menghormati layout yang ditetapkan oleh pass Transpiler sebelumnya (atau menggunakan trivial layout jika belum diatur). Biasanya hanya digunakan ketika Circuit harus dijalankan pada qubit tertentu di perangkat. Mode ini sering menghasilkan hasil yang lebih buruk karena ruang optimasinya lebih terbatas.
- `improve`: Mode ini menggunakan layout yang ditetapkan oleh pass Transpiler sebelumnya sebagai titik awal. Berguna ketika kamu memiliki tebakan awal yang baik untuk layout; misalnya, untuk Circuit yang dibangun dengan cara yang kurang lebih mengikuti coupling map perangkat. Mode ini juga berguna jika kamu ingin mencoba pass layout tertentu lainnya dikombinasikan dengan pass `AIRouting`.
- `optimize`: Ini adalah mode default. Paling cocok untuk Circuit umum di mana kamu mungkin tidak memiliki tebakan layout yang baik. Mode ini mengabaikan pemilihan layout sebelumnya.
- `local_mode`: Flag ini menentukan di mana pass `AIRouting` dijalankan. Jika `False`, `AIRouting` berjalan secara remote melalui Qiskit Transpiler Service. Jika `True`, paket akan mencoba menjalankan pass di lingkungan lokal kamu dengan fallback ke mode cloud jika dependensi yang diperlukan tidak ditemukan.

## Pass sintesis Circuit AI
Pass sintesis Circuit AI memungkinkan kamu untuk mengoptimalkan potongan-potongan dari berbagai jenis Circuit ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) dengan cara men-sintesis ulang. Cara umum untuk menggunakan pass sintesis adalah sebagai berikut: